In [1]:
!pip install flask pyngrok youtube-transcript-api pypdf pytubefix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 MB 14.9 MB/s eta 0:00:00


# ***Only to use llms***

In [ ]:
!pip install langchain
!pip install langchain-core
!pip install langchain-community

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

In [ ]:
!pip install colab-xterm

In [ ]:
%load_ext colabxterm

In [ ]:
%xterm

curl -fsSL https://ollama.com/install.sh | sh
ollama serve & ollama run codellama
ollama serve & ollama run mistral

In [ ]:
from langchain_community.llms import Ollama
h = Ollama(model="mistral")
code = Ollama(model="codellama")

# **From Here For main code or flask start**

In [2]:
from pyngrok import ngrok

ngrok.set_auth_token("37h1sYbGhtvgwpdZfd910NvgN1n_6nha5YkWSJrHYPNbHsiYh")

In [3]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)


In [4]:
%%writefile templates/home.html
<!DOCTYPE html>
<html>
<head>
    <title>YouTube & PDF Summarizer</title>
    <style>
* { box-sizing: border-box; }

body {
    font-family: Arial, sans-serif;
    background: #0f172a;
    color: #e5e7eb;
    margin: 0;
    padding: 40px 0;
}

.container {
    width: 90%;
    max-width: 850px;
    margin: auto;
    padding: 30px;
    background: #020617;
    border-radius: 16px;
    box-shadow: 0 20px 40px rgba(0,0,0,0.6);
}

h1 {
    text-align: center;
    margin-top: 0;
}

label {
    font-weight: 600;
}

input, select {
    width: 100%;
    padding: 10px 12px;
    margin: 6px 0 16px;
    border-radius: 8px;
    border: 1px solid #4b5563;
    background: #020617;
    color: #e5e7eb;
}

input:focus, select:focus {
    outline: none;
    border-color: #60a5fa;
    box-shadow: 0 0 0 1px #60a5fa;
}

button {
    padding: 10px 16px;
    border-radius: 999px;
    border: none;
    cursor: pointer;
    font-weight: 600;
    background: #1d4ed8;
    color: #f9fafb;
    transition: transform 0.1s ease, box-shadow 0.1s ease, background 0.2s ease;
    margin-bottom: 10px;
}

button:hover {
    background: #2563eb;
    box-shadow: 0 10px 18px rgba(37, 99, 235, 0.4);
    transform: translateY(-1px);
}

button:active {
    transform: translateY(0);
    box-shadow: none;
}

.section {
    margin-top: 20px;
    padding: 16px;
    border-top: 1px solid #1f2937;
    background: transparent;
}

.flash {
    background: #b91c1c;
    color: #fee2e2;
    padding: 10px 12px;
    border-radius: 8px;
    margin-bottom: 12px;
}

pre {
    width: 100%;
    background: #020617;
    color: #e5e7eb;
    border-radius: 8px;
    border: 1px solid #4b5563;
    padding: 10px;
    white-space: pre-wrap;
    word-wrap: break-word;
    max-height: 400px;
    overflow-y: auto;
}
</style>
</head>
<body>

<div class="container">
    <h1>YouTube & PDF Summarizer</h1>

    {% with messages = get_flashed_messages() %}
      {% for msg in messages %}
        <div class="flash">{{ msg }}</div>
      {% endfor %}
    {% endwith %}

    <form method="POST" enctype="multipart/form-data">

        <label>Select Input Type</label>
        <select id="inputType" name="input_type" onchange="toggleInput()">
            <option value="">-- Select Type --</option>

            <option value="youtube"
                {% if input_type == "youtube" %}selected{% endif %}>
                YouTube URL
            </option>

            <option value="pdf"
                {% if input_type == "pdf" %}selected{% endif %}>
                PDF File
            </option>
        </select>

        <!-- YOUTUBE SECTION -->
        <div id="youtubeInput" style="display:none;">

            <label>YouTube URL</label>
            <input type="text" name="url" placeholder="Enter YouTube URL"
                   value="{{ url or '' }}">

            <button type="submit" name="action" value="load">
                Load Transcripts
            </button>

            {% if languages %}
                <label>Select Transcript Language</label>
                <select name="language">
                    <option value="">-- Select Language --</option>
                    {% for code, name in languages %}
                        <option value="{{ code }}"
                            {% if selected_lang == code %}selected{% endif %}>
                            {{ name }} ({{ code }})
                        </option>
                    {% endfor %}
                </select>

                <button type="submit" name="action" value="summarize">
                    Summarize
                </button>
            {% endif %}
        </div>

        <!-- PDF SECTION -->
        <div id="pdfInput" style="display:none;">
            <label>Upload PDF</label>
            <input type="file" name="pdf_file" accept=".pdf">
            <button type="submit">Process & Summarize</button>
        </div>

        {% if transcript %}
        <div class="section">
            <h3>Summary Output</h3>
            <pre>{{ transcript }}</pre>
        </div>
        {% endif %}

    </form>
</div>

<script>
function toggleInput() {
    var type = document.getElementById("inputType").value;

    document.getElementById("youtubeInput").style.display = "none";
    document.getElementById("pdfInput").style.display = "none";

    if (type === "youtube") {
        document.getElementById("youtubeInput").style.display = "block";
    }

    if (type === "pdf") {
        document.getElementById("pdfInput").style.display = "block";
    }
}

// AUTO RESTORE UI AFTER REFRESH
window.onload = function() {
    toggleInput();
};
</script>

</body>
</html>

Writing templates/home.html


# **Code Type Detection**

In [5]:
import re
def detect_content_type(text: str) -> str:
    text_lower = text.lower()
    coding_score = 0
    theory_score = 0

    lines = text.split("\n")

    # -------------------------
    # 1️⃣ Python Control Keywords
    # -------------------------
    python_keywords = [
        "while", "for", "if", "elif", "else",
        "break", "continue", "pass",
        "print", "return"
    ]

    for word in python_keywords:
        matches = re.findall(rf'\b{word}\b', text_lower)
        coding_score += len(matches) * 3

    # -------------------------
    # 2️⃣ Assignment Detection
    # -------------------------
    for line in lines:
        if re.search(r'^\s*\w+\s*=\s*.+', line):
            coding_score += 4

    # -------------------------
    # 3️⃣ Indentation Block Detection (Python style)
    # -------------------------
    indented_lines = 0
    for line in lines:
        if re.match(r'^\s{2,}\S+', line):  # at least 2 spaces
            indented_lines += 1

    if indented_lines >= 2:
        coding_score += 5

    # -------------------------
    # 4️⃣ Code Symbols Check
    # -------------------------
    if ":" in text:
        coding_score += 3

    if "(" in text and ")" in text:
        coding_score += 3

    # -------------------------
    # 5️⃣ Natural Sentence Check
    # -------------------------
    sentences = re.split(r'[.!?]', text)
    valid_sentences = [s for s in sentences if s.strip()]

    if valid_sentences:
        avg_sentence_length = sum(len(s.split()) for s in valid_sentences) / len(valid_sentences)
        if avg_sentence_length > 12:
            theory_score += 5

    # -------------------------
    # 6️⃣ Final Decision
    # -------------------------
    if coding_score >= 6:
        return "Coding Content"
    else:
        return "Theory Content"

In [ ]:
result = detect_content_type(user_input)
if result == "Coding Content":
  pass
elif result == "Theory Content":
  pass
else:
  pass

# **Main Code**

In [6]:

import os
os.makedirs("templates", exist_ok=True)


from flask import Flask, render_template, request, flash
from pyngrok import ngrok
from pytubefix import YouTube
from pypdf import PdfReader
import re
import threading

# from langchain_community.llms import Ollama

ngrok.kill()


app = Flask(__name__)
app.secret_key = "finalyearproject2026"

def simple_summarizer(text):
    return text[:]

# ===============================
# MAIN ROUTE
# ===============================
@app.route("/", methods=["GET", "POST"])
def home():

    transcript = None
    languages = []
    url = None
    selected_lang = None
    input_type = None

    if request.method == "POST":

        input_type = request.form.get("input_type")
        url = request.form.get("url")
        selected_lang = request.form.get("language")
        action = request.form.get("action")
        pdf_file = request.files.get("pdf_file")

        try:

            # ===============================
            # YOUTUBE MODE
            # ===============================
            if input_type == "youtube" and url:

                yt = YouTube(url)

                # LOAD TRANSCRIPTS
                if action == "load":

                    if yt.captions:
                        languages = [(cap.code, cap.name) for cap in yt.captions]
                    else:
                        flash("Transcripts are not available for this video.")

                # SUMMARIZE OR FULL TRANSCRIPT
                elif action in ["summarize", "full"]:

                    if yt.captions:
                        languages = [(cap.code, cap.name) for cap in yt.captions]

                        if selected_lang:
                            caption = yt.captions.get(selected_lang)

                            if caption:
                                srt = caption.generate_srt_captions()

                                # Clean timestamps & numbers
                                srt = re.sub(r"\d{2}:\d{2}:\d{2},\d{3} --> .*?\n", "", srt)
                                srt = re.sub(r"^\d+\n", "", srt, flags=re.MULTILINE)

                                clean_text = " ".join(srt.split())

                                # If user selects FULL transcript
                                if action == "full":
                                    transcript = clean_text
                                    # here we can summarize the text not this
                                    result = detect_content_type(transcript)
                                    transcript = result

                                else:
                                    # here we can summarize the text
                                    transcript = simple_summarizer(clean_text)
                                    result = detect_content_type(transcript)
                                    if result == "Coding Content":
                                      transcript = code.invoke(transcript+"summarize the given content in simple english if there is code give me errors and solve the code")
                                    elif result == "Theory Content":
                                      transcript = h.invoke(transcript+"summarize the given content in simple english if there is code give me errors and solve the code")
                                    else:
                                      transcript = h.invoke(transcript+"summarize the given content in simple english if there is code give me errors and solve the code")


                            else:
                                flash("Selected language not found.")
                        else:
                            flash("Please select a language.")
                    else:
                        flash("Transcripts are not available for this video.")

            # ===============================
            # PDF MODE
            # ===============================
            elif input_type == "pdf" and pdf_file:

                reader = PdfReader(pdf_file)
                text = ""

                for page in reader.pages:
                    extracted = page.extract_text()
                    if extracted:
                        text += extracted + " "

                if action == "full":

                    transcript = text
                    # Here we needs to summarize not this
                    result = detect_content_type(transcript)
                    transcript = result

                else:
                    # here we can summarize the text
                    transcript = simple_summarizer(text)
                    result = detect_content_type(transcript)
                    if result == "Coding Content":
                      transcript = code.invoke(transcript+"summarize the given content in simple english if there is code give me errors and solve the code")
                    elif result == "Theory Content":
                      transcript = h.invoke(transcript+"summarize the given content in simple english if there is code give me errors and solve the code")
                    else:
                      transcript = h.invoke(transcript+"summarize the given content in simple english if there is code give me errors and solve the code")

        except Exception as e:
            flash("Error occurred. Check Colab output.")
            print("ERROR:", e)

    return render_template(
        "home.html",
        transcript=transcript,
        languages=languages,
        url=url,
        selected_lang=selected_lang,
        input_type=input_type
    )

# ===============================
# START NGROK + FLASK
# ===============================
public_url = ngrok.connect(5000)
print("🔥 Public URL:", public_url)

def run():
    app.run(port=5000)

threading.Thread(target=run).start()

🔥 Public URL: NgrokTunnel: "https://haggardly-unobeyed-germaine.ngrok-free.dev" -> "http://localhost:5000"
